In [1]:
import sys
print(sys.version)
print(sys.executable)

3.14.6 (v3.14.6:c63aec69bd5, Jun 10 2026, 08:07:54) [Clang 21.0.0 (clang-2100.1.1.101)]
/Library/Frameworks/Python.framework/Versions/3.14/bin/python3.14


In [2]:
%pip install pygame-ce


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:

import pygame
import random
import math

pygame.init()

WIDTH = 900
HEIGHT = 600
FPS = 60

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("AI vs AI Battle Arena")

clock = pygame.time.Clock()

font = pygame.font.SysFont("Arial", 25)
small_font = pygame.font.SysFont("Arial", 17)

BLACK = (18, 22, 30)
WHITE = (240, 240, 240)
BLUE = (60, 150, 255)
RED = (255, 75, 75)
GREEN = (70, 230, 130)
YELLOW = (255, 210, 60)
GRAY = (80, 85, 100)


pygame-ce 2.5.8 (SDL 2.32.10, Python 3.14.6)


In [4]:

class Fighter:

    def __init__(self, x, y, color, name, strategy):
        self.x = float(x)
        self.y = float(y)
        self.color = color
        self.name = name
        self.strategy = strategy

        self.radius = 18
        self.speed = 2.5
        self.health = 100
        self.cooldown = 0
        self.action = "Starting"

    def update(self, opponent, bullets):

        dx = opponent.x - self.x
        dy = opponent.y - self.y
        distance = max(1, math.hypot(dx, dy))

        ux = dx / distance
        uy = dy / distance

        # Movement perpendicular to the opponent
        sx = -uy
        sy = ux

        # Aggressive AI
        if self.strategy == "aggressive":

            if distance > 180:
                mx, my = ux, uy
                self.action = "Chasing"

            elif distance < 100:
                mx, my = -ux, -uy
                self.action = "Retreating"

            else:
                mx, my = sx, sy
                self.action = "Strafing"

        # Defensive AI
        else:

            if distance < 150:
                mx, my = -ux, -uy
                self.action = "Dodging"

            elif distance > 260:
                mx, my = ux, uy
                self.action = "Approaching"

            else:
                direction = random.choice([-1, 1])
                mx = sx * direction
                my = sy * direction
                self.action = "Defending"

        # Normalize movement
        length = max(1, math.hypot(mx, my))

        self.x += mx / length * self.speed
        self.y += my / length * self.speed

        # Keep fighter inside the arena
        self.x = max(20, min(WIDTH - 20, self.x))
        self.y = max(20, min(HEIGHT - 20, self.y))

        # Automatic shooting
        self.cooldown -= 1

        if self.cooldown <= 0 and distance < 500:
            bullets.append(
                Bullet(
                    self.x, self.y,
                    opponent.x, opponent.y,
                    self
                )
            )
            self.cooldown = 35

    def draw(self):

        pygame.draw.circle(
            screen,
            self.color,
            (int(self.x), int(self.y)),
            self.radius
        )

        pygame.draw.circle(
            screen,
            WHITE,
            (int(self.x), int(self.y)),
            self.radius,
            2
        )

        # Health bar
        bar_width = 70
        bar_x = int(self.x - bar_width / 2)
        bar_y = int(self.y - 32)

        pygame.draw.rect(
            screen, RED,
            (bar_x, bar_y, bar_width, 7)
        )

        health_width = int(
            bar_width * max(0, self.health) / 100
        )

        pygame.draw.rect(
            screen, GREEN,
            (bar_x, bar_y, health_width, 7)
        )


In [5]:

class Bullet:

    def __init__(self, x, y, target_x, target_y, owner):

        self.x = x
        self.y = y
        self.owner = owner

        self.radius = 5
        self.speed = 7
        self.damage = 10

        dx = target_x - x
        dy = target_y - y

        distance = max(1, math.hypot(dx, dy))

        self.vx = dx / distance * self.speed
        self.vy = dy / distance * self.speed

    def update(self):

        self.x += self.vx
        self.y += self.vy

    def draw(self):

        pygame.draw.circle(
            screen,
            self.owner.color,
            (int(self.x), int(self.y)),
            self.radius
        )


In [6]:

def reset_game():

    fighter1 = Fighter(
        150, 300,
        BLUE,
        "Blue AI",
        "aggressive"
    )

    fighter2 = Fighter(
        750, 300,
        RED,
        "Red AI",
        "defensive"
    )

    bullets = []

    return fighter1, fighter2, bullets


ai1, ai2, bullets = reset_game()

game_over = False
winner = None
running = True


In [7]:

while running:

    clock.tick(FPS)
    screen.fill(BLACK)

    # Handle window events
    for event in pygame.event.get():

        if event.type == pygame.QUIT:
            running = False

        if event.type == pygame.KEYDOWN:

            if event.key == pygame.K_r and game_over:
                ai1, ai2, bullets = reset_game()
                game_over = False
                winner = None

    # Update both AI fighters
    if not game_over:

        ai1.update(ai2, bullets)
        ai2.update(ai1, bullets)

        # Update bullets and check collisions
        for bullet in bullets[:]:

            bullet.update()

            # Remove bullets outside the window
            if not (
                0 <= bullet.x <= WIDTH
                and 0 <= bullet.y <= HEIGHT
            ):
                bullets.remove(bullet)
                continue

            # Find the opponent of the shooter
            target = ai2 if bullet.owner == ai1 else ai1

            distance = math.hypot(
                bullet.x - target.x,
                bullet.y - target.y
            )

            # Bullet hits the opponent
            if distance < bullet.radius + target.radius:

                target.health -= bullet.damage
                bullets.remove(bullet)

                if target.health <= 0:
                    target.health = 0
                    winner = bullet.owner.name
                    game_over = True

    # Draw the arena divider
    pygame.draw.line(
        screen,
        GRAY,
        (WIDTH // 2, 0),
        (WIDTH // 2, HEIGHT),
        1
    )

    # Draw fighters
    ai1.draw()
    ai2.draw()

    # Draw bullets
    for bullet in bullets:
        bullet.draw()

    # Display fighter names
    screen.blit(
        font.render("BLUE AI", True, BLUE),
        (25, 20)
    )

    screen.blit(
        font.render("RED AI", True, RED),
        (WIDTH - 150, 20)
    )

    # Display health and current actions
    screen.blit(
        small_font.render(
            f"Health: {ai1.health} | {ai1.action}",
            True, WHITE
        ),
        (25, 55)
    )

    screen.blit(
        small_font.render(
            f"Health: {ai2.health} | {ai2.action}",
            True, WHITE
        ),
        (WIDTH - 270, 55)
    )

    # Display the winner
    if game_over:

        overlay = pygame.Surface(
            (WIDTH, HEIGHT),
            pygame.SRCALPHA
        )

        overlay.fill((0, 0, 0, 180))
        screen.blit(overlay, (0, 0))

        message = font.render(
            winner + " WINS!",
            True, YELLOW
        )

        screen.blit(
            message,
            (
                WIDTH // 2 - message.get_width() // 2,
                HEIGHT // 2 - 30
            )
        )

        restart = small_font.render(
            "Press R to restart or close the window to exit",
            True, WHITE
        )

        screen.blit(
            restart,
            (
                WIDTH // 2 - restart.get_width() // 2,
                HEIGHT // 2 + 15
            )
        )

    pygame.display.flip()

pygame.quit()
